# MONZA x jpt - Analise essencial de resultados

Este notebook gera apenas os graficos essenciais do experimento MONZA runtime:

- Acuracia global federada por round dos CCs selecionados (`cc=2`, `cc=3`, `cc=7`).
- FPR/FRR por round dos CCs selecionados (`cc=2`, `cc=3`, `cc=7`).
- Recall por tipo de ataque nos CCs.
- Foco separado em `malicious_label`.

As colunas sao normalizadas para `DetectionFPR`/`DetectionFRR` (deteccao por-round, paper Eq 14/15 -- metrica headline) e `QuarantineFPR`/`QuarantineFRR` (ocupacao de quarentena, diagnostico que faz snowball por causa da quarentena exponencial). CSV antigo (`FPR`/`FRR`/`UploadFPR`/`UploadFRR`) e mapeado automaticamente. Quando existe `RunID`, usa o ultimo run completo.


In [ ]:
import json
import os
from pathlib import Path

import h5py
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

JPT = Path(os.environ.get('REPO_ROOT', Path.cwd())).resolve()
MONZA = JPT / 'PFLlibMonza' / 'system'
RESULTS_DIR = JPT / 'PFLlibMonza' / 'results'
DATASET_NAME = os.environ.get('DATASET_NAME', 'MNIST')
DATASET_SLUG = DATASET_NAME.lower()
ANALYSIS_OUT = Path(os.environ.get('ANALYSIS_OUT', JPT / f'artifacts/runs/{DATASET_SLUG}/notebook-analysis'))
ANALYSIS_OUT.mkdir(parents=True, exist_ok=True)
for stale_plot in ANALYSIS_OUT.glob('plot_*.png'):
    stale_plot.unlink()

EXPECTED_ROWS = int(os.environ.get('MONZA_EXPECTED_ROWS', '51'))
TAIL_ROUNDS = int(os.environ.get('MONZA_TAIL_ROUNDS', '30'))

plt.rcParams['figure.dpi'] = 110
plt.rcParams['font.size'] = 10

print('JPT:', JPT)
print('MONZA:', MONZA)
print('RESULTS_DIR:', RESULTS_DIR)
print('analysis output:', ANALYSIS_OUT)
print('EXPECTED_ROWS:', EXPECTED_ROWS, '| TAIL_ROUNDS:', TAIL_ROUNDS)


## Helpers de carga

As funcoes abaixo escolhem o ultimo run completo quando existe `RunID`; para CSV legado, usam o ultimo trecho em que `Round` reiniciou.


In [ ]:
def savefig(name: str):
    analysis_path = ANALYSIS_OUT / name
    plt.savefig(analysis_path, dpi=160, bbox_inches='tight')
    return analysis_path


def latest_run_by_round_reset(df: pd.DataFrame, min_rows: int = 1) -> pd.DataFrame:
    rounds = df['Round'].astype(int).tolist()
    starts = [0]
    prev = rounds[0] if rounds else 0
    for idx, r in enumerate(rounds[1:], start=1):
        if r < prev:
            starts.append(idx)
        prev = r
    for start in reversed(starts):
        candidate = df.iloc[start:].reset_index(drop=True)
        if len(candidate) >= min_rows:
            return candidate
    return df.iloc[starts[-1]:].reset_index(drop=True) if starts else df.copy()


def latest_run(df: pd.DataFrame, min_rounds: int = 1) -> pd.DataFrame:
    if df.empty:
        return df.copy()
    if 'RunID' in df.columns:
        run_ids = list(df['RunID'].drop_duplicates())
        for run_id in reversed(run_ids):
            candidate = df[df['RunID'] == run_id].reset_index(drop=True)
            if candidate['Round'].astype(int).nunique() >= min_rounds:
                return candidate
        return df[df['RunID'] == run_ids[-1]].reset_index(drop=True)
    return latest_run_by_round_reset(df, min_rows=min_rounds)


# Map legacy names -> canonical: Detection* (per-round, paper) and Quarantine* (snapshot).
_FPR_ALIASES = {
    'UploadFPR': 'DetectionFPR', 'UploadFRR': 'DetectionFRR',
    'FPR': 'QuarantineFPR', 'FRR': 'QuarantineFRR',
    'False Positive Rate': 'QuarantineFPR', 'False Rejection Rate': 'QuarantineFRR',
}


def load_fpr_frr_csv(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    df.columns = [c.strip() for c in df.columns]
    rename = {o: n for o, n in _FPR_ALIASES.items() if o in df.columns and n not in df.columns}
    df = df.rename(columns=rename)
    if 'Round' not in df.columns or 'DetectionFPR' not in df.columns:
        raise ValueError(f'{path} sem colunas Round/DetectionFPR (apos normalizacao)')
    keep = ['Round', 'DetectionFPR', 'DetectionFRR', 'QuarantineFPR', 'QuarantineFRR']
    keep = [c for c in keep if c in df.columns]
    if 'RunID' in df.columns:
        keep = ['RunID'] + keep
    df = df[keep].copy()
    df = df[df['Round'].astype(str) != 'Round'].reset_index(drop=True)
    df['Round'] = df['Round'].astype(int)
    for c in ['DetectionFPR', 'DetectionFRR', 'QuarantineFPR', 'QuarantineFRR']:
        if c in df.columns:
            df[c] = df[c].astype(float)
    return latest_run(df, min_rounds=EXPECTED_ROWS)



def find_latest_h5_for_cc(cc: int, dataset: str = DATASET_NAME) -> Path | None:
    if not RESULTS_DIR.exists():
        return None
    candidates = sorted(
        RESULTS_DIR.glob(f'{dataset}_FedAvg_{cc}_*_test_*.h5'),
        key=lambda p: p.stat().st_mtime,
    )
    return candidates[-1] if candidates else None


def load_accuracy_h5(path: Path) -> pd.DataFrame:
    with h5py.File(path, 'r') as h5:
        if 'rs_test_acc' not in h5:
            raise ValueError(f'{path.name} sem dataset rs_test_acc')
        acc = np.asarray(h5['rs_test_acc'], dtype=float)
    if acc.size == 0:
        raise ValueError(f'{path.name} tem rs_test_acc vazio')
    return pd.DataFrame({'Round': np.arange(acc.size), 'Accuracy': acc})


## 1. Detection FPR/FRR por round (paper Eq 14/15; tracejado = ocupacao de quarentena)


In [ ]:
DEFENSES = {
    'cc=3 (cosseno+score)': MONZA / 'fpr_frr_results_3.csv',
    'cc=7 (MLP+features)': MONZA / 'fpr_frr_results_7.csv',
}

COLORS = {
    'cc=3 (cosseno+score)': '#ff7f0e',
    'cc=7 (MLP+features)': '#d62728',
}

dfs = {}
missing = []
for name, path in DEFENSES.items():
    if not path.exists():
        missing.append(path.name)
        continue
    try:
        dfs[name] = load_fpr_frr_csv(path)
    except Exception as exc:
        print(f'Ignorando {path.name}: {exc}')

if not dfs:
    raise FileNotFoundError(f'Nenhum CSV fpr_frr_results_*.csv valido encontrado em {MONZA}')

for name, df in dfs.items():
    run_id = df['RunID'].iloc[0] if 'RunID' in df.columns else 'legacy'
    print(f'{name:45s} | run={run_id} | {len(df):3d} rows | DetectionFPR last={df.DetectionFPR.iloc[-1]:.4f} | DetectionFRR last={df.DetectionFRR.iloc[-1]:.4f}')
if missing:
    print('CSVs ausentes:', ', '.join(missing))


In [ ]:
rows = []
for name, df in dfs.items():
    tail = df.tail(TAIL_ROUNDS)
    rows.append({
        'Defesa': name,
        'Rows': len(df),
        'TailRows': len(tail),
        'DetectionFPR_mean': tail['DetectionFPR'].mean(),
        'DetectionFPR_std': tail['DetectionFPR'].std(),
        'DetectionFRR_mean': tail['DetectionFRR'].mean(),
        'DetectionFRR_std': tail['DetectionFRR'].std(),
        'QuarantineFPR_mean': tail['QuarantineFPR'].mean() if 'QuarantineFPR' in tail else float('nan'),
        'Score (Detection FPR+FRR)': tail['DetectionFPR'].mean() + tail['DetectionFRR'].mean(),
    })
summary = pd.DataFrame(rows).sort_values('Score (Detection FPR+FRR)').reset_index(drop=True)
summary


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5), sharex=True)
for name, df in dfs.items():
    color = COLORS.get(name, '#333333')
    # Headline: per-round detection rate (paper Eq 14/15).
    axes[0].plot(df['Round'], df['DetectionFPR'], label=name, color=color, linewidth=2)
    axes[1].plot(df['Round'], df['DetectionFRR'], label=name, color=color, linewidth=2)
    # Diagnostic: quarantine-occupancy snapshot (dashed).
    if 'QuarantineFPR' in df.columns:
        axes[0].plot(df['Round'], df['QuarantineFPR'], color=color, linewidth=1, linestyle='--', alpha=0.5)
        axes[1].plot(df['Round'], df['QuarantineFRR'], color=color, linewidth=1, linestyle='--', alpha=0.5)
axes[0].set_title('Detection FPR por round (tracejado = ocupacao de quarentena)')
axes[0].set_xlabel('Round')
axes[0].set_ylabel('DetectionFPR')
axes[0].grid(True, alpha=0.3)
axes[0].legend(loc='upper left', fontsize=8)
axes[0].set_ylim(-0.01, max(0.30, axes[0].get_ylim()[1]))
axes[1].set_title('Detection FRR por round (tracejado = ocupacao de quarentena)')
axes[1].set_xlabel('Round')
axes[1].set_ylabel('DetectionFRR')
axes[1].grid(True, alpha=0.3)
axes[1].legend(loc='upper left', fontsize=8)
axes[1].set_ylim(-0.01, max(0.60, axes[1].get_ylim()[1]))
plt.tight_layout()
savefig('plot_fpr_frr_by_round.png')


## 2. Acuracia global federada

Carrega `rs_test_acc` dos arquivos `.h5` em `PFLlibMonza/results/` apenas para `cc=2`, `cc=3` e `cc=7`.


In [ ]:
ACCURACY_DEFENSES = {
    'cc=2 (cluster cosseno)': 2,
    'cc=3 (cosseno+score)': 3,
    'cc=7 (MLP+features)': 7,
}

accuracy_frames = []
accuracy_summary_rows = []
for name, cc in ACCURACY_DEFENSES.items():
    h5_path = find_latest_h5_for_cc(cc)
    if h5_path is None:
        print(f'H5 ausente para {name}')
        continue
    try:
        acc_df = load_accuracy_h5(h5_path)
    except Exception as exc:
        print(f'Ignorando {h5_path.name}: {exc}')
        continue
    acc_df['Defesa'] = name
    acc_df['CC'] = cc
    acc_df['Arquivo'] = h5_path.name
    accuracy_frames.append(acc_df)
    accuracy_summary_rows.append({
        'Defesa': name,
        'CC': cc,
        'Arquivo': h5_path.name,
        'Rounds': len(acc_df),
        'BestAccuracy': float(acc_df['Accuracy'].max()),
        'FinalAccuracy': float(acc_df['Accuracy'].iloc[-1]),
    })

if accuracy_frames:
    accuracy_df = pd.concat(accuracy_frames, ignore_index=True)
    accuracy_summary = pd.DataFrame(accuracy_summary_rows).sort_values('CC').reset_index(drop=True)
else:
    accuracy_df = pd.DataFrame(columns=['Round', 'Accuracy', 'Defesa', 'CC', 'Arquivo'])
    accuracy_summary = pd.DataFrame(columns=['Defesa', 'CC', 'Arquivo', 'Rounds', 'BestAccuracy', 'FinalAccuracy'])

accuracy_summary


In [ ]:
if accuracy_df.empty:
    print('Nenhum .h5 de acuracia global encontrado em', RESULTS_DIR)
else:
    fig, ax = plt.subplots(figsize=(14, 6))
    for name, group in accuracy_df.groupby('Defesa', sort=False):
        ax.plot(
            group['Round'],
            group['Accuracy'],
            label=name,
            color=COLORS.get(name, '#333333'),
            linewidth=2,
        )
    ax.set_title('Acuracia global federada por round')
    ax.set_xlabel('Round')
    ax.set_ylabel('Acuracia de teste')
    ax.set_ylim(0.0, 1.02)
    ax.grid(True, alpha=0.3)
    ax.legend(loc='lower right', fontsize=8)
    plt.tight_layout()
    savefig('plot_global_accuracy_by_round.png')


## 3. FPR/recall por tipo de ataque nos CCs

Usa `cc_type_results_*.csv` quando existirem. Este notebook filtra apenas `cc=2`, `cc=3` e `cc=7`.


In [ ]:
def load_cc_type(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    if df.empty:
        return df
    df.columns = [c.strip() for c in df.columns]
    required = {'Round', 'CC', 'AttackType', 'Total', 'Removed', 'Rate', 'Metric'}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f'{path.name} sem colunas: {sorted(missing)}')
    df = latest_run(df, min_rounds=min(TAIL_ROUNDS, EXPECTED_ROWS))
    for col in ['Round', 'CC', 'Total', 'Removed']:
        df[col] = df[col].astype(int)
    df['Rate'] = df['Rate'].astype(float)
    return df

cc_type_frames = []
for cc in [2, 3, 7]:
    path = MONZA / f'cc_type_results_{cc}.csv'
    if not path.exists():
        print(f'CSV ausente: {path.name}')
        continue
    try:
        df = load_cc_type(path)
        if not df.empty:
            cc_type_frames.append(df)
    except Exception as exc:
        print(f'Ignorando {path.name}: {exc}')

if cc_type_frames:
    cc_type = pd.concat(cc_type_frames, ignore_index=True)
    cc_type['Defense'] = 'cc=' + cc_type['CC'].astype(str)
else:
    cc_type = pd.DataFrame()
    print('Nenhum cc_type_results_{2,3,7}.csv encontrado ainda. Rode cc=2/3/7 para gerar.')
cc_type.head()


In [ ]:
if cc_type.empty:
    cc_summary = pd.DataFrame()
else:
    rows = []
    for cc, cc_group in cc_type.groupby('CC', sort=True):
        tail_round_values = sorted(cc_group['Round'].astype(int).unique())[-TAIL_ROUNDS:]
        tail_cc = cc_group[cc_group['Round'].isin(tail_round_values)]
        for attack_type, group in tail_cc.groupby('AttackType', sort=True):
            total = group['Total'].sum()
            removed = group['Removed'].sum()
            rows.append({
                'CC': cc,
                'Defense': f'cc={cc}',
                'AttackType': attack_type,
                'Total': int(total),
                'Removed': int(removed),
                'Rate': float(removed / total) if total else 0.0,
                'Metric': 'FPR_upload_benign' if attack_type == 'benign' else 'recall',
            })
    cc_summary = pd.DataFrame(rows)
cc_summary


In [ ]:
if cc_summary.empty:
    print('Sem resumo por tipo de ataque dos CCs para plotar.')
else:
    order = ['benign', 'malicious_label', 'malicious_zeros', 'malicious_random', 'malicious_shuffle']
    pivot = cc_summary.pivot_table(index='AttackType', columns='Defense', values='Rate', aggfunc='mean')
    pivot = pivot.reindex([x for x in order if x in pivot.index] + [x for x in pivot.index if x not in order])
    fig, ax = plt.subplots(figsize=(12, 5))
    pivot.plot.bar(ax=ax, width=0.78)
    ax.axhline(0.05, color='gray', linestyle=':', linewidth=1.3, label='FPR target 5%')
    ax.set_title('FPR em uploads benignos e recall por tipo de ataque nos CCs')
    ax.set_ylabel('Taxa')
    ax.set_xlabel('')
    ax.set_ylim(0, max(1.0, float(pivot.max().max()) * 1.15))
    ax.set_xticklabels(pivot.index, rotation=20, ha='right')
    ax.grid(axis='y', alpha=0.25)
    ax.legend()
    plt.tight_layout()
    savefig('plot_cc_recall_by_attack_type.png')

    label = cc_summary[cc_summary['AttackType'] == 'malicious_label'].sort_values('CC')
    if not label.empty:
        fig, ax = plt.subplots(figsize=(8, 4))
        ax.bar(label['Defense'], label['Rate'], color='#8c564b')
        ax.set_title('Recall dos CCs em malicious_label')
        ax.set_ylabel('Recall')
        ax.set_ylim(0, max(1.0, float(label['Rate'].max()) * 1.2))
        ax.grid(axis='y', alpha=0.25)
        for idx, value in enumerate(label['Rate']):
            ax.text(idx, value + 0.02, f'{value:.2f}', ha='center', va='bottom')
        plt.tight_layout()
        savefig('plot_cc_malicious_label_recall.png')

    cc_summary.to_csv(ANALYSIS_OUT / 'cc_attack_type_summary.csv', index=False)
    cc_summary



## 4. Diagnostico por Tipo de Ataque

Usa `cc_detail_results_{2,3,7}.csv` quando existir. Mostra decisões por upload, tipo de ataque e score do baseline/detector.


In [ ]:

detail_frames = []
for cc in [2, 3, 7]:
    path = MONZA / f'cc_detail_results_{cc}.csv'
    if path.exists():
        frame = pd.read_csv(path)
        frame.columns = [c.strip() for c in frame.columns]
        frame = latest_run(frame, min_rounds=min(TAIL_ROUNDS, EXPECTED_ROWS))
        detail_frames.append(frame)

if detail_frames:
    detail = pd.concat(detail_frames, ignore_index=True)
    tail_rounds = sorted(detail['Round'].astype(int).unique())[-TAIL_ROUNDS:]
    detail_tail = detail[detail['Round'].astype(int).isin(tail_rounds)].copy()
else:
    detail = pd.DataFrame()
    detail_tail = pd.DataFrame()
    print('cc_detail_results_{2,3,7}.csv ausentes. Rode cc=2/3/7 para gerar diagnostico dos detectores e baselines.')

detail_tail.head()
